In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load preprocessed expression data
combined_raw = pd.read_csv("combined_raw.csv", index_col=0)
combined_zscored = pd.read_csv("combined_zscored.csv", index_col=0)

print(f"Combined raw: {combined_raw.shape}")
print(f"Combined zscored: {combined_zscored.shape}")

# Should both be 10394 genes * 550 patients

Combined raw: (10394, 550)
Combined zscored: (10394, 550)


In [4]:
# Loading somatic mutation data for later causal mapping

mutations = pd.read_csv(r"D:\School\IITD\General\GBM_new\gbm_tcga\data_mutations.txt", sep="\t", low_memory=False)
#print(f"Shape: {mutations.shape}")
#print(f"Columns: {mutations.columns.tolist()}")
#print(f"\nFirst row:")
#print(mutations.iloc[0])

# Shape: 22073 * 90
# Columns: ['Hugo_Symbol', 'Entrez_Gene_Id', 'Variant_Classification'. 'Tumor_Sample_Barcode'...] are the key columns needed
# Specifically the sample barcode matches patient ID


In [6]:
# Check mutation types
print("Variant classifications:")
print(mutations["Variant_Classification"].value_counts())

# Check patient barcode format
print("\nExample barcodes:")
print(mutations["Tumor_Sample_Barcode"].iloc[:5].tolist())

# Check total unique patients (it is 290)

Variant classifications:
Variant_Classification
Missense_Mutation         13805
Silent                     5194
Nonsense_Mutation           837
Frame_Shift_Del             555
RNA                         486
Splice_Site                 359
In_Frame_Del                211
Frame_Shift_Ins             207
Splice_Region               174
Intron                       59
5'UTR                        59
3'Flank                      42
In_Frame_Ins                 28
5'Flank                      24
Nonstop_Mutation             11
Translation_Start_Site       11
3'UTR                        10
IGR                           1
Name: count, dtype: int64

Example barcodes:
['TCGA-76-6193-01', 'TCGA-76-6193-01', 'TCGA-76-6193-01', 'TCGA-76-6193-01', 'TCGA-76-6193-01']


In [7]:
# Paper: "missense, nonsense, frameshift, in-frame insertion or deletion, 
# splice site, modified translation start site, introduced new start site, or removed stop codon"
coding_mutations = [
    "Missense_Mutation",
    "Nonsense_Mutation", 
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Site",
    "Translation_Start_Site",
    "Nonstop_Mutation"
]

# Filter to coding mutations only
mutations_filtered = mutations[mutations["Variant_Classification"].isin(coding_mutations)]
print(f"Total mutations before filter: {len(mutations)}")
print(f"Total mutations after filter: {len(mutations_filtered)}")

# Strip barcode to patient level
mutations_filtered = mutations_filtered.copy()
mutations_filtered["patient_id"] = mutations_filtered["Tumor_Sample_Barcode"].apply(
    lambda x: "-".join(x.split("-")[:3])
)

print(f"\nUnique patients in mutations: {mutations_filtered['patient_id'].nunique()}")
print(f"Example patient ID: {mutations_filtered['patient_id'].iloc[0]}")

Total mutations before filter: 22073
Total mutations after filter: 16024

Unique patients in mutations: 290
Example patient ID: TCGA-76-6193


In [18]:
# Build binary mutation matrix
# First filter to patients in our expression data
expression_patients = set(combined_zscored.columns.tolist())
mutations_filtered = mutations_filtered[mutations_filtered["patient_id"].isin(expression_patients)]

#print(f"Mutations from patients in expression data: {len(mutations_filtered)}")
#print(f"Unique patients after intersection: {mutations_filtered['patient_id'].nunique()}")

# Build binary matrix
mutation_matrix = mutations_filtered.groupby(["Hugo_Symbol", "patient_id"]).size().unstack(fill_value=0)
mutation_matrix = (mutation_matrix > 0).astype(int)  # binarize - 1 if any mutation, 0 if none

#print(f"\nMutation matrix shape: {mutation_matrix.shape}")

# Apply 5% frequency filter
# Paper: "Mutated genes and pathways used in causal network inference 
# were required to be mutated in at least 5% of the patient tumors"
n_patients = mutation_matrix.shape[1]
min_patients = int(0.05 * n_patients)

#print(f"\n5% of {n_patients} patients = {min_patients} patients minimum")

mutation_freq = mutation_matrix.sum(axis=1)
mutation_matrix_filtered = mutation_matrix[mutation_freq >= min_patients]

#print(f"Genes before 5% filter: {mutation_matrix.shape[0]}")
#print(f"Genes after 5% filter: {mutation_matrix_filtered.shape[0]}")
#print(f"\nTop mutated genes:")
#print(mutation_freq.sort_values(ascending=False).head(10))

In [16]:
# Info for reference:

# Mutations from patients in expression data: 14405
# Unique patients after intersection: 259
# Initial mutation matrix: 7645 genes * 259 patients

# After keeping 5% filter as specificed by paper: 35 genes * 259 patients
# Rows: genes, columns: 1/0 depending on mutated/wildtype in each patient

# Top mutated genes: PTEN, TP53, EFGR, TIN, MUC16, PIK3CA, FLG, RYR2, PIK3R1, NF1

In [19]:
mutation_matrix_filtered.to_csv("mutation_matrix.csv")
print(f"Saved mutation_matrix.csv")
print(f"Shape: {mutation_matrix_filtered.shape}")
print(f"Genes: {mutation_matrix_filtered.index.tolist()}")

Saved mutation_matrix.csv
Shape: (35, 259)
Genes: ['AHNAK2', 'APOB', 'ATRX', 'CNTNAP2', 'COL6A3', 'DNAH2', 'DNAH5', 'EGFR', 'FLG', 'HMCN1', 'IDH1', 'KEL', 'KRTAP4-11', 'LRP2', 'MUC16', 'MUC17', 'NBPF10', 'NF1', 'NLRP5', 'OBSCN', 'PCLO', 'PIK3CA', 'PIK3R1', 'PKHD1', 'PTEN', 'RB1', 'RELN', 'RYR2', 'RYR3', 'SPTA1', 'SYNE1', 'TCHH', 'TP53', 'TTN', 'USH2A']


In [24]:
# Now loading clinical data

clinical = pd.read_csv(r"D:\School\IITD\General\GBM_new\gbm_tcga\data_clinical_patient.txt", sep="\t", comment="#", index_col=0)
#print(f"Shape: {clinical.shape}")
#print(f"Columns: {clinical.columns.tolist()}")
#print(f"\nFirst row:")
#print(clinical.iloc[0])

# Shape is 596 patients * 37 params
# Columns: ['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', ...] ; these are the relevant ones

# Extract relevant columns
clinical_filtered = clinical[["PATIENT_ID", "OS_STATUS", "OS_MONTHS"]].copy()

# Recode OS_STATUS to binary
clinical_filtered["OS_STATUS"] = clinical_filtered["OS_STATUS"].map({
    "1:DECEASED": 1,
    "0:LIVING": 0
})

# Convert OS_MONTHS to numeric, coerce errors to NaN
clinical_filtered["OS_MONTHS"] = pd.to_numeric(clinical_filtered["OS_MONTHS"], errors="coerce")

#print(f"OS_STATUS after recoding:")
#print(clinical_filtered["OS_STATUS"].value_counts())
#print(f"\nOS_MONTHS after conversion:")
#print(clinical_filtered["OS_MONTHS"].describe())
#print(f"\nMissing values after conversion:")
#print(clinical_filtered.isna().sum())

# Set PATIENT_ID as index
clinical_filtered = clinical_filtered.set_index("PATIENT_ID")

# Intersect with expression patients
expression_patients = combined_zscored.columns.tolist()
clinical_filtered = clinical_filtered.loc[clinical_filtered.index.isin(expression_patients)]
print(f"\nPatients after intersecting with expression data: {clinical_filtered.shape[0]}")


Patients after intersecting with expression data: 547


In [ ]:
# For reference: 

'''

OS_STATUS
1.0    492
0.0    102
where 1 == DECEASED and 0 == ALIVE/ CENSORED

Patients after intersecting with expression data: 547

'''


In [25]:
clinical_filtered.to_csv("clinical_data.csv")
print(f"Saved clinical_data.csv")

print(f"Shape: {clinical_filtered.shape}")

Saved clinical_data.csv
Shape: (547, 2)


In [43]:
all_mirna = {}

for uuid in os.listdir(mirna_folder):
    uuid_path = os.path.join(mirna_folder, uuid)
    
    # Skip non-folders (manifest files etc.)
    if not os.path.isdir(uuid_path):
        continue
    
    txt_files = [f for f in os.listdir(uuid_path) if f.endswith(".txt")]
    if not txt_files:
        continue
    
    txt_path = os.path.join(uuid_path, txt_files[0])
    
    try:
        df = pd.read_csv(txt_path, sep="\t", index_col=0)
        
        # Skip metadata files that GDC bundles alongside expression files
        # These have columns like 'submitter_id', 'entity_type' instead of expression values
        if "reads_per_million_miRNA_mapped" not in df.columns:
            continue
        
        # RPM = reads per million miRNA mapped
        # Normalizes for sequencing depth - patients sequenced more deeply
        # would have higher raw counts regardless of biology
        # RPM makes counts comparable across samples
        # Key: folder UUID is the GDC file ID, used later for TCGA barcode mapping
        all_mirna[uuid] = df["reads_per_million_miRNA_mapped"]
        
    except Exception as e:
        print(f"Error in {uuid}: {e}")

print(f"Loaded {len(all_mirna)} samples")

mirna_matrix = pd.DataFrame(all_mirna)

# dropna() keeps only miRNAs detected in ALL 266 samples
# Ensures complete matrix with no missing values for downstream correlation analysis
mirna_matrix = mirna_matrix.dropna()
print(f"Matrix shape: {mirna_matrix.shape}")
# Expected: (1881, 266)

Loaded 266 samples
Matrix shape: (1881, 266)


In [44]:
# Map UUIDs to TCGA barcodes, filter to primary tumor
uuids = mirna_matrix.columns.tolist()
uuid_to_barcode = {}
batch_size = 200

for i in range(0, len(uuids), batch_size):
    batch = uuids[i:i+batch_size]
    payload = {
        "filters": {
            "op": "in",
            "content": {"field": "file_id", "value": batch}
        },
        "fields": "file_id,cases.submitter_id,cases.samples.sample_type",
        "size": len(batch)
    }
    response = requests.post("https://api.gdc.cancer.gov/files", json=payload)
    for hit in response.json()["data"]["hits"]:
        file_id = hit["file_id"]
        case_id = hit["cases"][0]["submitter_id"]
        samples = hit["cases"][0].get("samples", [])
        primary = [s for s in samples if s["sample_type"] == "Primary Tumor"]
        if primary:
            uuid_to_barcode[file_id] = case_id
    print(f"Processed {min(i+batch_size, len(uuids))}/{len(uuids)}")

print(f"Mapped: {len(uuid_to_barcode)} primary tumor samples")

# Filter to primary tumor only then rename
mirna_matrix = mirna_matrix.loc[:, mirna_matrix.columns.isin(uuid_to_barcode.keys())]
mirna_matrix.columns = [uuid_to_barcode[col] for col in mirna_matrix.columns]
print(f"Shape after filtering: {mirna_matrix.shape}")

# Handle duplicates
dupes = mirna_matrix.columns[mirna_matrix.columns.duplicated(keep=False)].unique().tolist()
print(f"Duplicate patients: {len(dupes)}")

if dupes:
    non_dup = mirna_matrix.loc[:, ~mirna_matrix.columns.isin(dupes)]
    best_samples = []
    for patient in dupes:
        cols = mirna_matrix.loc[:, mirna_matrix.columns == patient]
        best = cols.iloc[:, cols.median(axis=0).argmax()]
        best.name = patient
        best_samples.append(best)
    best_dup = pd.concat(best_samples, axis=1)
    mirna_matrix = pd.concat([non_dup, best_dup], axis=1)

print(f"Final shape: {mirna_matrix.shape}")

# log2(RPM+1)
mirna_log = np.log2(mirna_matrix + 1)

# Remove zero variance miRNAs
zero_var = mirna_log.var(axis=1) == 0
print(f"Zero variance miRNAs: {zero_var.sum()}")
mirna_log = mirna_log.loc[~zero_var]

# Z-score
mirna_zscored = pd.DataFrame(
    stats.zscore(mirna_log.values, axis=1),
    index=mirna_log.index,
    columns=mirna_log.columns
)

print(f"\nAfter Z-scoring:")
print(f"Shape: {mirna_zscored.shape}")
print(f"Min: {mirna_zscored.min().min():.2f}")
print(f"Max: {mirna_zscored.max().max():.2f}")
print(f"Mean: {mirna_zscored.mean().mean():.4f}")
print(f"NaN: {mirna_zscored.isna().sum().sum()}")

# Save
mirna_log.to_csv("mirna_log2rpm.csv")
mirna_zscored.to_csv("mirna_zscored.csv")
print("\nSaved mirna_log2rpm.csv and mirna_zscored.csv")

Processed 200/266
Processed 266/266
Mapped: 260 primary tumor samples
Shape after filtering: (1881, 260)
Duplicate patients: 5
Final shape: (1881, 255)
Zero variance miRNAs: 278

After Z-scoring:
Shape: (1603, 255)
Min: -7.46
Max: 15.94
Mean: -0.0000
NaN: 0

Saved mirna_log2rpm.csv and mirna_zscored.csv


In [46]:
# For reference: Shape: (1603, 255) (miRNAs * patients)
# No NaN or inf values
# 245 out of these 255 patients have expression data and miRNA data